# vLLM で大量のプロンプトをまとめて高速に生成する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/14_llm/vllm_generation.ipynb)

NLP クックブック 14-04 ／ 大規模言語モデル (LLM)

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

この章の「複数のプロンプトをまとめてバッチ生成する」よりも多くの入力を、もっと速く処理したい。推論エンジンの vLLM で手元のモデルを動かし、transformers の `generate` と速度と出力を比べたい。

In [1]:
# vLLM 0.34.0 は torch 2.14.0 を指定しているので、Colab の torch 2.14.1 が入れ替わる（Colab で 2〜4 分。pip が依存関係の衝突を表示するが、このレシピは動く）
# 新しいランタイムで、torch を import する前に実行する（すでに import していたらランタイムを再起動する）
!pip install -q vllm==0.34.0

# 依存ライブラリ (nvidia-cutlass-dsl) は .pth ファイルで import の検索パスを足すが、.pth は Python の起動時にしか読まれない。
# このままだと vLLM の起動中に「No module named 'cutlass'」で止まるので、ランタイムを再起動するか、次の 2 行で読み直す
import site
for d in site.getsitepackages() + [site.getusersitepackages()]: site.addsitedir(d)

In [2]:
import gc, json, shutil, time
import pandas as pd
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
import vllm
from transformers import AutoModelForCausalLM, AutoTokenizer

print("torch", torch.__version__, "/ vLLM", vllm.__version__)
torch.manual_seed(0)
name = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"   # SB Intuitions、MIT ライセンス
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
tokenizer.padding_side = "left"
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

# この章の「複数のプロンプトをまとめてバッチ生成する」の 8 語を 16 語に増やし、4 種類の質問と組み合わせた 64 件
words = ["形態素解析", "機械翻訳", "固有表現抽出", "要約", "質問応答", "感情分析", "音声認識", "情報検索",
         "構文解析", "単語埋め込み", "言語モデル", "文書分類", "対話システム", "文字認識", "照応解析", "かな漢字変換"]
templates = ["「{}」を一文で説明してください。", "「{}」の具体的な使い道を一つ挙げてください。",
             "「{}」を小学生にもわかるように説明してください。", "「{}」の難しいところは何ですか。簡潔に答えてください。"]
conversations = [[{"role": "user", "content": t.format(w)}] for w in words for t in templates]
prompts = [tokenizer.apply_chat_template(c, add_generation_prompt=True, tokenize=False) for c in conversations]
MAX_NEW = 128
REP = 1.05   # モデルの generation_config.json の値。transformers は自動で使うが、vLLM は SamplingParams で指定しないと使わない

@torch.no_grad()
def hf_generate(batch):
    """貪欲法で生成し、生成したトークン ID の列（パディングと文末記号を除く）を返す"""
    enc = tokenizer(batch, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, repetition_penalty=REP, pad_token_id=tokenizer.pad_token_id)
    return [[t for t in row if t not in (tokenizer.pad_token_id, tokenizer.eos_token_id)] for row in out[:, enc["input_ids"].shape[1]:].tolist()]

results = {}
def record(label, outs, seconds):
    n_tok = sum(len(o) for o in outs)
    results[label] = {"件数": len(outs), "秒": seconds, "1 件あたりの秒": seconds / len(outs), "出力トークン/秒": n_tok / seconds}

hf_generate(prompts[:2])                                 # 1 回目は準備に時間がかかるので、計測の前に動かしておく
torch.cuda.synchronize(); t = time.perf_counter()
hf_single = [hf_generate([p])[0] for p in prompts[:8]]   # 1 件ずつは時間がかかるので先頭の 8 件だけ
torch.cuda.synchronize(); record("transformers 1 件ずつ", hf_single, time.perf_counter() - t)
t = time.perf_counter()
hf_batch = hf_generate(prompts)                          # 64 件を 1 回のバッチで
torch.cuda.synchronize(); record("transformers バッチ", hf_batch, time.perf_counter() - t)
print(pd.DataFrame(results).T.round(2).to_string())
print(f"GPU メモリ最大 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

torch 2.14.0+cu130 / vLLM 0.34.0
                      件数      秒  1 件あたりの秒  出力トークン/秒
transformers 1 件ずつ   8.0  13.41      1.68     63.60
transformers バッチ    64.0   2.49      0.04   2662.01
GPU メモリ最大 2.3 GB


In [3]:
from vllm import LLM, SamplingParams
from vllm.tokenizers import get_tokenizer

def gpu_memory():
    """GPU 全体の使用量 (GB)。Colab では GPU を 1 人で使うので、このノートブックと vLLM のエンジン（子プロセス）の合計になる"""
    free, total = torch.cuda.mem_get_info()
    return (total - free) / 1e9

del model; gc.collect(); torch.cuda.empty_cache()        # transformers のモデルを GPU から外して、vLLM に場所を空ける

# vLLM も transformers の AutoTokenizer でトークナイザーを読むので、同じ不具合が起きる
s = "東京都千代田区の国立国会図書館に行った。"
print("vLLM が既定で読むトークナイザー:", get_tokenizer(name).encode(s, add_special_tokens=False))
print("SentencePiece（学習時と同じ分割）:", sp.encode(s))

# 直したトークナイザーを保存して vLLM に渡す。AutoTokenizer で読み直すと tokenizer_config.json の
# LlamaTokenizer として組み立て直されてまた壊れるので、tokenizer.json をそのまま使うクラスに書き換える
tok_dir = "sarashina_tokenizer"
tokenizer.save_pretrained(tok_dir)
cfg = json.load(open(f"{tok_dir}/tokenizer_config.json"))
cfg["tokenizer_class"] = "PreTrainedTokenizerFast"
json.dump(cfg, open(f"{tok_dir}/tokenizer_config.json", "w"), ensure_ascii=False, indent=1)
shutil.copy(hf_hub_download(name, "config.json"), tok_dir)   # ないと vLLM がこのディレクトリの config.json を探して ERROR を表示する（動作は変わらない）

t = time.perf_counter()
llm = LLM(name, tokenizer=tok_dir,
          dtype="bfloat16" if BF16 else "float16",   # T4 は bfloat16 に非対応なので float16 にする
          max_model_len=2048,                        # 1 件の最大の長さ（プロンプト + 出力）。既定はモデルの上限 (8192)
          gpu_memory_utilization=0.6,                # GPU メモリ全体のうち vLLM が使う割合（モデル、CUDA グラフ、KV キャッシュ）。既定は 0.9
          seed=0)
print(f"起動 {time.perf_counter() - t:.0f} 秒、GPU メモリ {gpu_memory():.1f} GB")   # torch.compile と CUDA グラフの準備をするので時間がかかる（Colab で 2〜5 分）
print("vLLM のトークナイザーの分割が SentencePiece と一致:", llm.get_tokenizer().encode(s, add_special_tokens=False) == sp.encode(s))

vLLM が既定で読むトークナイザー: [4044, 54771, 27440, 13466, 20305, 9565, 9119, 278]
SentencePiece（学習時と同じ分割）: [41241, 291, 13466, 20305, 9565, 9119, 278]
(MainProcess pid=1587620) INFO 11-17 17:25:48 [startup.py:96] non-default args: {'tokenizer': 'sarashina_tokenizer', 'dtype': 'float16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.38, 'disable_log_stats': True, 'attention_backend': 'TRITON_ATTN', 'model': 'sbintuitions/sarashina2.2-0.5b-instruct-v0.1'}
(MainProcess pid=1587620) INFO 11-17 17:26:02 [model.py:761] Resolved architecture: LlamaForCausalLM
(MainProcess pid=1587620) WARNING 11-17 17:26:02 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=1587620) INFO 11-17 17:26:02 [model.py:2214] Using max model len 2048
(MainProcess pid=1587620) INFO 11-17 17:26:02 [scheduler.py:354] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=1587620) INFO 11-17 17:26:02 [kernel.py:517] Final IR op priority after setting platform defaults: IrOpPriorit

In [4]:
# 貪欲法 (temperature=0)。max_tokens の既定は 16 なので必ず指定する
greedy = SamplingParams(temperature=0, max_tokens=MAX_NEW, repetition_penalty=REP)
ids = lambda outs: [[t for t in o.outputs[0].token_ids if t != tokenizer.eos_token_id] for o in outs]

llm.chat(conversations[:2], greedy, use_tqdm=False)     # 1 回目は準備に時間がかかるので、計測の前に動かしておく
t = time.perf_counter()
vl_single = [ids(llm.chat([c], greedy, use_tqdm=False))[0] for c in conversations[:8]]
record("vLLM 1 件ずつ", vl_single, time.perf_counter() - t)
t = time.perf_counter()
outs = llm.chat(conversations, greedy, use_tqdm=False)  # llm.chat はチャットテンプレートを当ててから生成する
vl_batch = ids(outs)
record("vLLM まとめて", vl_batch, time.perf_counter() - t)
print(pd.DataFrame(results).T.round(2).to_string())

# 入力: llm.chat が作ったトークン列は、transformers の apply_chat_template と同じか
enc = [tokenizer(p, add_special_tokens=False)["input_ids"] for p in prompts]
print("\nプロンプトのトークン列が一致:", sum(o.prompt_token_ids == e for o, e in zip(outs, enc)), "/", len(outs))
# 出力: transformers のバッチ生成と vLLM の出力が一致するか。違う場合は何トークン目から分かれたか
def prefix(a, b):
    return next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))
same = [a == b for a, b in zip(hf_batch, vl_batch)]
split = [prefix(a, b) for a, b, s in zip(hf_batch, vl_batch, same) if not s]
print(f"出力が完全に一致: {sum(same)} / {len(same)} 件。一致しない {len(split)} 件で分かれた位置（トークン目）: {sorted(split)}")
print(f"transformers の 1 件ずつとバッチの一致: {sum(a == b for a, b in zip(hf_single, hf_batch))} / 8 件、"
      f"vLLM の 1 件ずつとまとめての一致: {sum(a == b for a, b in zip(vl_single, vl_batch))} / 8 件")
diff = [k for k, s in enumerate(same) if not s]
i = diff[0] if diff else 0
print("\n例:", conversations[i][0]["content"])
print("  transformers:", tokenizer.decode(hf_batch[i]))
print("  vLLM        :", outs[i].outputs[0].text)

                      件数      秒  1 件あたりの秒  出力トークン/秒
transformers 1 件ずつ   8.0  13.41      1.68     63.60
transformers バッチ    64.0   2.49      0.04   2662.01
vLLM 1 件ずつ           8.0   2.37      0.30    360.45
vLLM まとめて           64.0   0.55      0.01  12059.02

プロンプトのトークン列が一致: 64 / 64
出力が完全に一致: 55 / 64 件。一致しない 9 件で分かれた位置（トークン目）: [10, 13, 32, 60, 81, 81, 85, 95, 111]
transformers の 1 件ずつとバッチの一致: 7 / 8 件、vLLM の 1 件ずつとまとめての一致: 6 / 8 件

例: 「形態素解析」を小学生にもわかるように説明してください。
  transformers: 「形態素解析」は、文章やテキストデータをコンピュータが理解しやすくするための重要な技術の一つです。これを小学生にもわかるように説明すると、次のようになります。

まず、文章には言葉や単語がたくさんあります。例えば、「こんにちは」、「世界平和」、「リンゴ」などがそうですね。これらの言葉はそれぞれ「人」、「地球」、「果実」といった特定の意味を持っています。しかし、これらの言葉を一度にすべて理解するのは複雑です。そこで、形態素解析というプロセスが役立ちます。

形態素解析とは、これらの言葉を細かく分割して、それぞれの意味や役割を理解するための作業を指します。具体的には、文章中の単語を「品詞」と呼ばれるカテゴリーに分類します。品
  vLLM        : 「形態素解析」は、文章やテキストデータをコンピュータが理解しやすくするための重要な技術の一つです。これを小学生にもわかるように説明すると、次のようになります。

まず、文章には言葉や単語がたくさんあります。例えば、「こんにちは」、「世界平和」、「リンゴ」などがそうですね。これらの言葉はそれぞれ「人」、「場所」、「もの」など、特定の役割を持っています。この役割のことを「

In [5]:
# 1 つのプロンプトから複数の候補を作る: n で本数、temperature と top_p で多様さ、seed で再現性を決める
sampling = SamplingParams(n=4, temperature=0.8, top_p=0.95, max_tokens=MAX_NEW, repetition_penalty=REP, seed=0)
llm.chat(conversations[:2], sampling, use_tqdm=False)   # 計測の前に動かしておく
t = time.perf_counter()
outs_n = llm.chat(conversations, sampling, use_tqdm=False)
seconds = time.perf_counter() - t
n_tok = sum(len(c.token_ids) for o in outs_n for c in o.outputs)
print(f"64 件 × 4 本 = {sum(len(o.outputs) for o in outs_n)} 本: {seconds:.1f} 秒、{n_tok / seconds:.0f} トークン/秒")
print("4 本がすべて違う文になったプロンプト:", sum(len({c.text for c in o.outputs}) == 4 for o in outs_n), "/", len(outs_n))
outs_n2 = llm.chat(conversations, sampling, use_tqdm=False)   # 同じ設定でもう一度
print("もう一度実行して 4 本とも同じだったプロンプト:",
      sum([c.text for c in a.outputs] == [c.text for c in b.outputs] for a, b in zip(outs_n, outs_n2)), "/", len(outs_n))
print("\n" + conversations[0][0]["content"])
for c in outs_n[0].outputs:
    print(" -", c.text.replace("\n", " ")[:80])
print(f"\nGPU メモリの使用量（このノートブックと vLLM のエンジン）: {gpu_memory():.1f} GB")

64 件 × 4 本 = 256 本: 2.6 秒、10180 トークン/秒
4 本がすべて違う文になったプロンプト: 64 / 64
もう一度実行して 4 本とも同じだったプロンプト: 64 / 64

「形態素解析」を一文で説明してください。
 - 「形態素解析」は、文を構成する単語や品詞に分割する処理のことを指します。これにより、文の意味をより正確に理解し、特定の単語や表現の関連性を分析することが可能にな
 - 「形態素解析」は、テキストを単語や品詞に分割する処理を指します。これにより、文章の意味解析や検索が容易になり、情報抽出や自動翻訳など、さまざまな用途に活用できま
 - 「形態素解析」は、文章中の単語やフレーズをその基本的な単位である「形態素」に分割し、それぞれの意味や文脈を特定する処理です。これにより、文章の意味を理解したり、
 - 「形態素解析」は、テキストデータ（特に日本語文）を意味や品詞ごとに分割する手法を指します。これにより、文章の意味を解析し、必要な情報を効率的に抽出することが可能

GPU メモリの使用量（このノートブックと vLLM のエンジン）: 10.1 GB


In [6]:
# enforce_eager=True: CUDA グラフを作らずに 1 演算ずつ実行する。起動は速いが、生成は遅くなる
del llm; gc.collect(); torch.cuda.empty_cache()          # 前のエンジンを止めて GPU を空ける
print(f"止めたあとの GPU メモリ: {gpu_memory():.1f} GB")
t = time.perf_counter()
llm = LLM(name, tokenizer=tok_dir, dtype="bfloat16" if BF16 else "float16", max_model_len=2048,
          gpu_memory_utilization=0.6, enforce_eager=True, seed=0)
startup = time.perf_counter() - t
llm.chat(conversations[:2], greedy, use_tqdm=False)
t = time.perf_counter()
vl_eager = ids(llm.chat(conversations, greedy, use_tqdm=False))
record("vLLM まとめて (enforce_eager)", vl_eager, time.perf_counter() - t)
print(f"起動 {startup:.0f} 秒、GPU メモリ {gpu_memory():.1f} GB")
print(pd.DataFrame(results).T.round(2).to_string())
print("CUDA グラフありとの出力の一致:", sum(a == b for a, b in zip(vl_batch, vl_eager)), "/", len(vl_eager))

(MainProcess pid=1587620) INFO 11-17 17:28:21 [utils.py:628] [shutdown] Process manager: send sigterm to process EngineCore
止めたあとの GPU メモリ: 0.3 GB
(MainProcess pid=1587620) INFO 11-17 17:28:25 [startup.py:96] non-default args: {'tokenizer': 'sarashina_tokenizer', 'dtype': 'float16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.38, 'disable_log_stats': True, 'enforce_eager': True, 'attention_backend': 'TRITON_ATTN', 'model': 'sbintuitions/sarashina2.2-0.5b-instruct-v0.1'}
(MainProcess pid=1587620) INFO 11-17 17:28:25 [model.py:761] Resolved architecture: LlamaForCausalLM
(MainProcess pid=1587620) WARNING 11-17 17:28:25 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=1587620) INFO 11-17 17:28:25 [model.py:2214] Using max model len 2048
(MainProcess pid=1587620) WARNING 11-17 17:28:25 [vllm.py:1709] Enforce eager set, disabling torch.compile, CUDAGraphs, and JIT kernel warmup. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none and --kernel_

- vLLM (Kwon et al., 2023) は、KV キャッシュをブロック単位で管理する PagedAttention と、終わった系列の空きにすぐ次の入力を入れる連続バッチ処理 (continuous batching) で、多数の入力を GPU にすき間なく流す推論エンジンである（Apache 2.0 ライセンス）。T4 (compute capability 7.5) は bfloat16 と FlashAttention に対応していないので `dtype="float16"` にし、アテンションは vLLM が自動で Triton の実装 (TRITON_ATTN) を選ぶ。上の出力は float16 と TRITON_ATTN を指定して T4 より速い別の GPU で実行したもので、T4 の実機では確かめていない。
- インストール: vLLM 0.34.0 は torch の版を固定しているので、Colab の torch 2.14.1 が 2.14.0 に入れ替わり、huggingface-hub なども上がる（検証の環境で 120 秒。pip が依存関係の衝突を表示する）。新しいランタイムで最初に実行すれば再起動は要らないが、依存ライブラリが .pth ファイルで足す検索パスを `site.addsitedir` で読み直す必要がある（ないと、エンジンの起動中に `No module named 'cutlass'` で止まった）。一般に、読み込み済みの torch は入れ替えられないので、先に torch を import していたらランタイムを再起動する。
- 速度（同じ 64 件、貪欲法、最大 128 トークン）: transformers は 1 件ずつで 1 件あたり 1.68 秒、64 件のバッチで 2,662 トークン/秒、vLLM は 1 件ずつで 0.30 秒、まとめて 12,059 トークン/秒（transformers のバッチの約 4.5 倍）だった。ただし vLLM の起動に 139 秒かかった（torch.compile と CUDA グラフの準備）ので、数十件を一度だけ処理するなら transformers のバッチ生成で足りる。一般に、件数が多く、出力の長さがばらつくほど vLLM が有利になる。時間は共有の GPU で測った目安で、T4 では遅くなる。
- 出力の一致: `llm.chat` が作ったプロンプトのトークン列は 64 件すべてで `apply_chat_template` と同じだったが、どちらも貪欲法なのに生成した文が完全に一致したのは 55 / 64 件で、残りは 10〜111 トークン目で分かれた（例では「地球」と「場所」）。float16 ではアテンションの実装やバッチの組み方で末尾の桁が変わり、確率がほぼ同じ候補の順位が入れ替わるためで、同じ方式でも 1 件ずつとまとめての一致は vLLM で 6 / 8 件、transformers で 7 / 8 件だった。vLLM に切り替えたら評価をやり直す。
- トークナイザー: vLLM も transformers の `AutoTokenizer` で読むので、sarashina2.2 では同じ不具合が起きる（上の出力の「東京都千代田区の」の分割）。直したトークナイザーを保存し、`tokenizer_config.json` のクラスを `PreTrainedTokenizerFast` に書き換えて `tokenizer=` に渡すと、SentencePiece と一致した。不具合の内容は、この章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きを参照。モデルは sarashina2.2-0.5b-instruct（SB Intuitions、MIT ライセンス）。
- `SamplingParams`: `max_tokens` の既定は 16 で、指定しないと出力が途中で切れる。`SamplingParams` を渡さないときだけモデルの generation_config.json（temperature 0.7、top_p 0.9、repetition_penalty 1.05）が既定値になるので、渡すときは必要な値をすべて書く。`n=4` では 256 本を 2.6 秒（10,180 トークン/秒）で作り、64 件すべてで 4 本が別の文になった。`seed=0` で同じ呼び出しをもう一度実行すると、この実行では 64 件すべてで同じ 4 本になったが、float16 の計算の揺れで変わることもありうるので、完全な再現は前提にしない。
- GPU メモリ: vLLM は起動時に `gpu_memory_utilization`（GPU 全体に対する割合。既定は 0.9）の分を先に確保し、モデルと CUDA グラフの残りを KV キャッシュにする。ここでは 0.6（T4 で約 9.5GB）にし、transformers のモデルは先に GPU から外した。検証では 24GB の GPU でほぼ同じ量になる 0.38 にし、このノートブックと vLLM のエンジンの分だけを数えると最大 10.1 GB、`del llm` の後は 0.3 GB だった。起動時に「No available memory for the cache blocks」で止まったら、`gpu_memory_utilization` を上げるか `max_model_len` を下げる。
- `enforce_eager=True` にすると、起動は 34 秒と短くなったが、生成は 2,129 トークン/秒と既定（CUDA グラフあり）の約 6 分の 1 になった。確保するメモリは変わらず（どちらも起動の直後に 9.8 GB）、CUDA グラフの分は KV キャッシュに回る。出力の一致は 53 / 64 件だった。試行錯誤の段階など、起動の速さを優先するときに使う。